# 버스노선(BusRoute) → DuckDB 적재

공공데이터포털 `BusRoute/getBusRoute` API.

- `opr_ymd` (운행일자): **오늘 - 1개월** (공공데이터 API 특성상 최근 데이터는 비어있을 수 있어 한 달 전 기준)
- `ctpv_cd` (시도 코드 2자리): `region` 테이블에서 전국 17개 시도를 순회
- 테이블: `bus_route` (PK = rte_id)
- 향후 `station` 테이블이 추가되면 dptre_sttn_id / arvl_sttn_id로 JOIN 가능

In [1]:
# ============================================================
# 셀 1 - 환경 설정 + DB 헬퍼 정의
# ============================================================
# 락 점유 최소화: db_open() 컨텍스트 매니저로 짧게 연결하고 try/finally로 반드시 닫음.

import os
import time
import requests
import duckdb
import pandas as pd
from pathlib import Path
from contextlib import contextmanager
from datetime import date
from dateutil.relativedelta import relativedelta
from urllib.parse import unquote
from dotenv import load_dotenv

load_dotenv()
SERVICE_KEY = unquote(os.environ["MY_SERVICE_KEY"])

DB_PATH = "data/seoul.duckdb"
Path("data").mkdir(exist_ok=True)


@contextmanager
def db_open(read_only: bool = False):
    """짧게 연결하고 무조건 닫는 컨텍스트 매니저 (예외 시에도 안전)."""
    con = duckdb.connect(DB_PATH, read_only=read_only)
    try:
        yield con
    finally:
        con.close()


print(f"SERVICE_KEY 로드됨 (길이 {len(SERVICE_KEY)})")
print(f"DB: {DB_PATH} (헬퍼 준비됨)")

SERVICE_KEY 로드됨 (길이 88)
DB: data/seoul.duckdb (헬퍼 준비됨)


In [2]:
# ============================================================
# 셀 2 - Pydantic 모델 (공공데이터 표준 응답 + BusRouteItem)
# ============================================================
# data_ingestion.ipynb에 있는 PublicDataResponse 구조와 동일.
# 노트북 간 import가 번거로워 여기서 재정의.
# (TODO: 공통화가 필요해지면 models.py로 추출)

from typing import Generic, TypeVar, Optional
from pydantic import BaseModel, field_validator

T = TypeVar("T", bound=BaseModel)


class Header(BaseModel):
    resultCode: str
    resultMsg: str


class Items(BaseModel, Generic[T]):
    item: list[T] = []

    @field_validator("item", mode="before")
    @classmethod
    def _coerce_to_list(cls, v):
        if v is None or v == "":
            return []
        if isinstance(v, dict):   # 1건일 때 dict로 오는 공공데이터 API 변동 대응
            return [v]
        return v


class Body(BaseModel, Generic[T]):
    items: Items[T] = Items[T]()
    dataType: str | None = None
    pageNo: int
    numOfRows: int
    totalCount: int


class Response_(BaseModel, Generic[T]):
    header: Header
    body: Body[T]


class PublicDataResponse(BaseModel, Generic[T]):
    Response: Response_[T]


class BusRouteItem(BaseModel):
    """버스노선 한 건.

    특이사항:
      - sgg_cd는 여기서 5자리 (sido(2) + 하위 3자리) → region 테이블의 3자리 sgg_cd와 자릿수가 다름
      - rte_id는 노선 전역 고유 ID → PK로 사용
      - dptre(출발) / arvl(도착) 정류소는 노선의 양 끝점만 표현 (경유지 아님)
      - ⚠️ API가 일부 노선의 rte_nm / rte_no / 정류소 필드를 null로 반환 → rte_id 외 모두 Optional
    """
    # 핵심 필드 (항상 값이 있어야 함)
    rte_id: str                                  # 노선 ID (PK 후보)
    opr_ymd: str                                 # 운행일자 YYYYMMDD
    ctpv_cd: str                                 # 시도 코드 (2자리)

    # Optional: API가 null로 내려줄 수 있는 필드들
    sgg_cd: Optional[str] = None                 # 시군구 코드 (5자리)
    rte_no: Optional[str] = None                 # 노선 번호
    rte_nm: Optional[str] = None                 # 노선 전체명
    dptre_sttn_id: Optional[str] = None          # 출발 정류소 ID
    dptre_sttn_nm: Optional[str] = None          # 출발 정류소명
    arvl_sttn_id: Optional[str] = None           # 도착 정류소 ID
    arvl_sttn_nm: Optional[str] = None           # 도착 정류소명


print("모델 로드 완료")

모델 로드 완료


In [3]:
# ============================================================
# 셀 3 - 공공데이터 표준 fetch (시도 + 날짜 파라미터 지원)
# ============================================================
# data_ingestion.ipynb의 fetch_all_pages와 동일 로직.
# BusRoute API는 opr_ymd, ctpv_cd를 extra_params로 받아 호출.
#
# 특이사항: 이 API는 해당 시도/날짜에 데이터가 없으면 다음 형태로 응답:
#   {"Error": {"code": "50", "message": "NO_DATA_FOUND"}}
# → Pydantic 검증 전에 Error 키 존재 여부를 선확인하고, 있으면 빈 리스트 반환.

BUS_ROUTE_URL = "https://apis.data.go.kr/1613000/BusRoute/getBusRoute"


def fetch_all_pages(
    url: str,
    item_model: type[T],
    extra_params: dict | None = None,
    num_rows: int = 1000,
) -> list[T]:
    """공공데이터 표준 응답 API 전체 페이지 수집.
    NO_DATA_FOUND 에러는 빈 리스트로 처리하고 다음 시도로 넘어감.
    """
    params_base = {
        "serviceKey": SERVICE_KEY,
        "numOfRows": num_rows,
        "dataType": "JSON",
        **(extra_params or {}),
    }
    all_items: list[T] = []
    page = 1
    while True:
        res = requests.get(url, params={**params_base, "pageNo": page}, timeout=30)
        res.raise_for_status()
        payload = res.json()

        # ── 에러 응답 선처리 ─────────────────────────────────
        # NO_DATA_FOUND 같은 정상 범주 에러는 빈 결과로 취급
        if "Error" in payload:
            err = payload["Error"]
            code = err.get("code")
            msg = err.get("message", "")
            if msg == "NO_DATA_FOUND" or code == "50":
                return all_items  # 해당 시도는 데이터 없음 → 여태 모은 거 반환
            raise RuntimeError(f"API error: {code} {msg}")

        # ── Pydantic 검증 ───────────────────────────────────
        parsed = PublicDataResponse[item_model].model_validate(payload)
        header = parsed.Response.header
        if header.resultCode not in ("00", "200"):
            raise RuntimeError(f"API error: {header.resultCode} {header.resultMsg}")

        body = parsed.Response.body
        items = body.items.item
        all_items.extend(items)

        if len(all_items) >= body.totalCount or not items:
            break
        page += 1
        time.sleep(0.15)

    return all_items


print("fetch 함수 준비 완료")

fetch 함수 준비 완료


In [4]:
# ============================================================
# 셀 4 - 파라미터 준비: opr_ymd + sido 목록
# ============================================================
# opr_ymd는 '오늘 - 1개월'
opr_ymd = (date.today() - relativedelta(months=1)).strftime("%Y%m%d")
print(f"운행일자 opr_ymd = {opr_ymd}")

# region 테이블에서 시도 목록 추출 (read_only 커넥션 — 다른 노트북이 쓰기 중이어도 안전)
with db_open(read_only=True) as con:
    sido_list_df = con.execute("""
        SELECT sido_cd, locatadd_nm AS sido_nm
        FROM region
        WHERE sgg_cd='000' AND umd_cd='000' AND ri_cd='00'
        ORDER BY sido_cd
    """).df()

print(f"\n시도 {len(sido_list_df)}개:")
print(sido_list_df)

운행일자 opr_ymd = 20260322

시도 16개:
   sido_cd  sido_nm
0       11    서울특별시
1       26    부산광역시
2       27    대구광역시
3       28    인천광역시
4       29    광주광역시
5       30    대전광역시
6       31    울산광역시
7       41      경기도
8       43     충청북도
9       44     충청남도
10      46     전라남도
11      47     경상북도
12      48     경상남도
13      50  제주특별자치도
14      51  강원특별자치도
15      52  전북특별자치도


In [5]:
# ============================================================
# 셀 5 - 전국 시도 순회하며 버스노선 수집
# ============================================================
# 각 시도별로 fetch_all_pages 호출 → 결과 누적.
# 시도마다 수백~수천 건 예상. 전체 완료까지 수 분 소요 가능.

all_routes: list[BusRouteItem] = []

for _, row in sido_list_df.iterrows():
    sido_cd = row["sido_cd"]
    sido_nm = row["sido_nm"]

    routes = fetch_all_pages(
        BUS_ROUTE_URL,
        BusRouteItem,
        extra_params={"opr_ymd": opr_ymd, "ctpv_cd": sido_cd},
    )
    all_routes.extend(routes)
    print(f"  [{sido_cd}] {sido_nm}: +{len(routes)}건 (누적 {len(all_routes)})")
    time.sleep(0.2)   # 시도 간 간격

# Pydantic → dict → DataFrame
bus_route_df = pd.DataFrame([r.model_dump() for r in all_routes])
print(f"\n총 {len(bus_route_df)}건 수집")
print(bus_route_df.head())

  [11] 서울특별시: +672건 (누적 672)
  [26] 부산광역시: +800건 (누적 1472)
  [27] 대구광역시: +361건 (누적 1833)
  [28] 인천광역시: +343건 (누적 2176)
  [29] 광주광역시: +117건 (누적 2293)
  [30] 대전광역시: +118건 (누적 2411)
  [31] 울산광역시: +920건 (누적 3331)
  [41] 경기도: +4232건 (누적 7563)
  [43] 충청북도: +1504건 (누적 9067)
  [44] 충청남도: +4199건 (누적 13266)
  [46] 전라남도: +2661건 (누적 15927)
  [47] 경상북도: +3075건 (누적 19002)
  [48] 경상남도: +3502건 (누적 22504)
  [50] 제주특별자치도: +674건 (누적 23178)
  [51] 강원특별자치도: +0건 (누적 23178)
  [52] 전북특별자치도: +156건 (누적 23334)

총 23334건 수집
     rte_id   opr_ymd ctpv_cd sgg_cd  rte_no                rte_nm  \
0  11110897  20260322      11  11000    0017  0017번(용산차고지신용산역3번출구)   
1  92000001  20260322      11  11000  한강버스01          한강버스01(마곡잠실)   
2  11110706  20260322      11  11000   영등포01   영등포01(신도림역구로디지털단지역)   
3  11110629  20260322      11  11000   서대문01      서대문01(삼성빌라홍제삼거리)   
4  11110590  20260322      11  11000   동대문01       동대문01(회기역경희의료원)   

  dptre_sttn_id dptre_sttn_nm arvl_sttn_id   arvl_sttn_nm  
0       8503228  

In [6]:
# ============================================================
# 셀 6 - DuckDB 적재 (bus_route 테이블)
# ============================================================
# 중복 rte_id 사전 정리 (시도 경계 노선이 양쪽에서 반환될 수 있음)
before = len(bus_route_df)
bus_route_df_dedup = bus_route_df.drop_duplicates(subset=["rte_id"])
after = len(bus_route_df_dedup)
if before != after:
    print(f"⚠️  중복 rte_id {before - after}건 제거 (시도 경계 노선일 가능성)")

with db_open() as con:
    con.execute("""
    CREATE TABLE IF NOT EXISTS bus_route (
        rte_id         TEXT PRIMARY KEY,  -- 노선 ID
        opr_ymd        TEXT,              -- 운행일자 YYYYMMDD
        ctpv_cd        TEXT,              -- 시도 코드 2자리
        sgg_cd         TEXT,              -- 시군구 코드 5자리 (시도 포함)
        rte_no         TEXT,              -- 노선 번호 (예: 영등포01)
        rte_nm         TEXT,              -- 노선 전체명
        dptre_sttn_id  TEXT,              -- 출발 정류소 ID
        dptre_sttn_nm  TEXT,              -- 출발 정류소명
        arvl_sttn_id   TEXT,              -- 도착 정류소 ID
        arvl_sttn_nm   TEXT               -- 도착 정류소명
    )
    """)

    con.execute("DELETE FROM bus_route")
    con.register("bus_route_view", bus_route_df_dedup)
    con.execute("""
    INSERT INTO bus_route
    SELECT rte_id, opr_ymd, ctpv_cd, sgg_cd,
           rte_no, rte_nm,
           dptre_sttn_id, dptre_sttn_nm, arvl_sttn_id, arvl_sttn_nm
    FROM bus_route_view
    """)
    con.unregister("bus_route_view")

    cnt = con.execute("SELECT COUNT(*) FROM bus_route").fetchone()[0]

print(f"적재 완료: {cnt}건")

⚠️  중복 rte_id 1378건 제거 (시도 경계 노선일 가능성)
적재 완료: 21956건


In [7]:
# ============================================================
# 셀 7 - 검증
# ============================================================
with db_open(read_only=True) as con:
    print("=== 시도별 버스 노선 수 ===")
    print(con.execute("""
        SELECT b.ctpv_cd,
               ANY_VALUE(r.locatadd_nm) AS sido_nm,
               COUNT(*) AS route_cnt
        FROM bus_route b
        LEFT JOIN region r
          ON r.sido_cd = b.ctpv_cd AND r.sgg_cd='000' AND r.umd_cd='000'
        GROUP BY b.ctpv_cd
        ORDER BY route_cnt DESC
    """).df())

    print("\n=== 서울 노선 샘플 10건 ===")
    print(con.execute("""
        SELECT rte_no, rte_nm, dptre_sttn_nm, arvl_sttn_nm
        FROM bus_route
        WHERE ctpv_cd='11'
        LIMIT 10
    """).df())

=== 시도별 버스 노선 수 ===
   ctpv_cd  sido_nm  route_cnt
0       41      경기도       4232
1       44     충청남도       3742
2       47     경상북도       3075
3       48     경상남도       2983
4       46     전라남도       2659
5       43     충청북도       1504
6       31    울산광역시        920
7       50  제주특별자치도        674
8       11    서울특별시        672
9       26    부산광역시        400
10      27    대구광역시        361
11      28    인천광역시        343
12      52  전북특별자치도        156
13      30    대전광역시        118
14      29    광주광역시        117

=== 서울 노선 샘플 10건 ===
   rte_no                 rte_nm dptre_sttn_nm   arvl_sttn_nm
0    0017   0017번(용산차고지신용산역3번출구)     용산차고지(가상)  용산역3번출구.용산차고지
1  한강버스01           한강버스01(마곡잠실)            마곡             잠실
2   영등포01    영등포01(신도림역구로디지털단지역)      신도림역2번출구       신도림역2번출구
3   서대문01       서대문01(삼성빌라홍제삼거리)          삼성빌라           삼성빌라
4   동대문01        동대문01(회기역경희의료원)           회기역            회기역
5    마포01  마포01(산천동리버힐삼성아파트용문시장)   산천동리버힐삼성아파트       원효2동주민센터
6    광진01    광진01(광진정보도서관워커힐